# Lab 02 - Keyword Search: TF-IDF and BM25

**Knowledge base:** `knowledge_base/03_retrieval/03_keyword_search.md`

**Concepts:** Bag-of-words · TF-IDF · BM25 · inverted index · term saturation · length normalization

BM25 is the dominant keyword search algorithm. This lab builds it from scratch
so you understand every component before using library implementations.

---

## Setup

In [1]:
import numpy as np
import math
from collections import Counter, defaultdict
from typing import List, Tuple

print("✅ Imports OK")

✅ Imports OK


---
## 1 - Bag of words

Both TF-IDF and BM25 treat documents as **bags of words** - word order is ignored,
only word counts matter. This is the simplest possible document representation.

In [2]:
documents = [
    "the cat sat on the mat",
    "the dog lay on the rug",
    "cats and dogs are popular pets",
    "the mat is on the floor near the rug",
    "cats sat on rugs and dogs sat on mats",
]

# Tokenize - split into lowercase words
def tokenize(text: str) -> List[str]:
    return text.lower().split()

# Show the bag-of-words representation
for doc in documents:
    tokens = tokenize(doc)
    counts = Counter(tokens)
    print(f"'{doc[:40]}...' -> {dict(counts)}")

'the cat sat on the mat...' -> {'the': 2, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1}
'the dog lay on the rug...' -> {'the': 2, 'dog': 1, 'lay': 1, 'on': 1, 'rug': 1}
'cats and dogs are popular pets...' -> {'cats': 1, 'and': 1, 'dogs': 1, 'are': 1, 'popular': 1, 'pets': 1}
'the mat is on the floor near the rug...' -> {'the': 3, 'mat': 1, 'is': 1, 'on': 1, 'floor': 1, 'near': 1, 'rug': 1}
'cats sat on rugs and dogs sat on mats...' -> {'cats': 1, 'sat': 2, 'on': 2, 'rugs': 1, 'and': 1, 'dogs': 1, 'mats': 1}


---
## 2 - TF-IDF

**Term Frequency (TF):** how often a word appears in *this document*, normalized by document length.

$$TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

**Inverse Document Frequency (IDF):** how rare the word is across *all documents*.
Rare words are more informative signals than common words.

$$IDF(t) = \log\frac{N}{|\{d : t \in d\}|}$$

**TF-IDF score:** TF × IDF

In [3]:
def compute_tf(doc_tokens: List[str]) -> dict:
    '''Term frequency for all words in a document.'''
    counts = Counter(doc_tokens)
    total  = len(doc_tokens)
    return {word: count / total for word, count in counts.items()}


def compute_idf(all_docs_tokens: List[List[str]]) -> dict:
    '''Inverse document frequency for all words across all documents.'''
    N  = len(all_docs_tokens)
    df = defaultdict(int)   # document frequency: how many docs contain each word
    for tokens in all_docs_tokens:
        for word in set(tokens):  # set so we count each word once per doc
            df[word] += 1
    return {word: math.log(N / count) for word, count in df.items()}


def tfidf_score(query: str, document: str, idf: dict) -> float:
    '''TF-IDF score for a query against a document.'''
    query_tokens = tokenize(query)
    doc_tokens   = tokenize(document)
    tf           = compute_tf(doc_tokens)
    score        = 0.0
    for term in query_tokens:
        if term in tf and term in idf:
            score += tf[term] * idf[term]
    return score


# Build the IDF table from all documents
all_tokens = [tokenize(doc) for doc in documents]
idf        = compute_idf(all_tokens)

print("IDF values (lower = more common, higher = rarer):")
for word, val in sorted(idf.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {word:15s}: {val:.4f}")

IDF values (lower = more common, higher = rarer):
  cat            : 1.6094
  dog            : 1.6094
  lay            : 1.6094
  pets           : 1.6094
  are            : 1.6094
  popular        : 1.6094
  is             : 1.6094
  near           : 1.6094
  floor          : 1.6094
  mats           : 1.6094


In [4]:
# Rank all documents for a query using TF-IDF
def retrieve_tfidf(query: str, docs: List[str], idf: dict, top_k: int = 3) -> List[Tuple]:
    scores = [(tfidf_score(query, doc, idf), doc) for doc in docs]
    return sorted(scores, reverse=True)[:top_k]


query = "cats sat on mats"
print(f"Query: '{query}'\n")
for score, doc in retrieve_tfidf(query, documents, idf):
    print(f"  [{score:.5f}] {doc}")

Query: 'cats sat on mats'

  [0.53384] cats sat on rugs and dogs sat on mats
  [0.18991] the cat sat on the mat
  [0.15272] cats and dogs are popular pets


---
## 3 - BM25: two improvements over TF-IDF

BM25 fixes two weaknesses:

**Problem 1 - Linear TF:** A document with the keyword 20 times scores exactly twice as high
as one with it 10 times. But relevance doesn't work that way - it saturates.

**BM25 fix - TF saturation:** The contribution of repeated occurrences diminishes.
Controlled by `k1`. At high repetition, the score approaches an upper bound instead of growing forever.

**Problem 2 - Length bias:** TF-IDF penalizes long documents because raw word counts are
divided by total words. A long document with many good keywords still scores low.

**BM25 fix - length normalization:** Applies a softer length penalty with a tunable `b` parameter.

In [5]:
class BM25:
    '''
    BM25 ranking algorithm from scratch.

    Parameters:
        k1 (float): Term saturation. Higher -> less saturation. Typical: 1.2-2.0.
        b  (float): Length normalization. 0 = off, 1 = full. Typical: 0.75.
    '''

    def __init__(self, documents: List[str], k1: float = 1.5, b: float = 0.75):
        self.k1   = k1
        self.b    = b
        self.docs = [tokenize(d) for d in documents]
        self.N    = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / self.N

        # Document frequency
        self.df = defaultdict(int)
        for doc in self.docs:
            for word in set(doc):
                self.df[word] += 1

        # IDF (BM25 variant - slightly different formula from classic TF-IDF)
        self.idf = {}
        for word, freq in self.df.items():
            self.idf[word] = math.log((self.N - freq + 0.5) / (freq + 0.5) + 1)

    def score(self, query: str, doc_idx: int) -> float:
        '''BM25 score for a single query against a single document.'''
        query_tokens = tokenize(query)
        doc_tokens   = self.docs[doc_idx]
        dl           = len(doc_tokens)
        tf_counts    = Counter(doc_tokens)
        score        = 0.0

        for term in query_tokens:
            if term not in self.idf:
                continue
            tf  = tf_counts.get(term, 0)
            # BM25 term weight formula - saturation applied here
            num = tf * (self.k1 + 1)
            den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += self.idf[term] * (num / den)

        return score

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple]:
        '''Retrieve top-k documents for a query.'''
        scores = [(self.score(query, i), self.docs[i]) for i in range(self.N)]
        scores.sort(reverse=True)
        return scores[:top_k]


# Instantiate and test
bm25  = BM25(documents)
query = "cats sat on mats"
print(f"Query: '{query}'\n")
results = bm25.retrieve(query, top_k=3)
for score, tokens in results:
    print(f"  [{score:.4f}] {' '.join(tokens)}")

Query: 'cats sat on mats'

  [3.5711] cats sat on rugs and dogs sat on mats
  [1.2575] the cat sat on the mat
  [0.9465] cats and dogs are popular pets


---
## 4 - TF saturation: seeing the BM25 improvement

BM25's most important improvement is that repeating a keyword doesn't keep adding score.
Let's see this directly.

In [6]:
# Create documents with different repetition counts of the same keyword
repetition_docs = [
    "cat",                      # 1 occurrence
    "cat cat cat cat cat",      # 5 occurrences
    "cat " * 20,                # 20 occurrences
    "cat " * 50,                # 50 occurrences
]

bm25_rep  = BM25(repetition_docs, k1=1.5, b=0.0)  # b=0 to disable length norm
query_rep = "cat"

print("Term frequency saturation comparison:")
print(f"{'Occurrences':>12}  {'TF-IDF':>10}  {'BM25':>10}")
print("-" * 38)

all_tokens_rep = [tokenize(d) for d in repetition_docs]
idf_rep = compute_idf(all_tokens_rep)

for i, doc in enumerate(repetition_docs):
    tfidf_s = tfidf_score(query_rep, doc, idf_rep)
    bm25_s  = bm25_rep.score(query_rep, i)
    count   = doc.strip().count("cat")
    print(f"  {count:>10}x  {tfidf_s:>10.4f}  {bm25_s:>10.4f}")

print()
print("TF-IDF score keeps growing - no saturation.")
print("BM25 score plateaus - diminishing returns on repetition.")

Term frequency saturation comparison:
 Occurrences      TF-IDF        BM25
--------------------------------------
           1x      0.0000      0.1054
           5x      0.0000      0.2026
          20x      0.0000      0.2450
          50x      0.0000      0.2557

TF-IDF score keeps growing - no saturation.
BM25 score plateaus - diminishing returns on repetition.


---
## 5 - Use rank-bm25 library

For production use, the `rank-bm25` library provides an optimized, well-tested BM25 implementation.

In [8]:
# Install rank-bm25 if needed
import subprocess
subprocess.run(["pip", "install", "rank-bm25", "--break-system-packages", "-q"], capture_output=True)

from rank_bm25 import BM25Okapi

# The library expects pre-tokenized input
tokenized_docs = [tokenize(d) for d in documents]
bm25_lib = BM25Okapi(tokenized_docs)

query     = "cats sat on mats"
query_tok = tokenize(query)
scores    = bm25_lib.get_scores(query_tok)

print(f"BM25 (rank-bm25 library) - Query: '{query}'")
ranked = sorted(zip(scores, documents), reverse=True)
for score, doc in ranked[:3]:
    print(f"  [{score:.4f}] {doc}")

BM25 (rank-bm25 library) - Query: 'cats sat on mats'
  [1.9553] cats sat on rugs and dogs sat on mats
  [0.5440] the cat sat on the mat
  [0.3638] cats and dogs are popular pets


---
## 6 - Exercise: build a BM25 search over a real document set

In [ ]:
# Exercise - rank the following documents for the given queries using rank-bm25

corpus = [
    "Python is a high-level programming language used for data science and AI.",
    "Machine learning models learn patterns from training data.",
    "Neural networks are inspired by the structure of the human brain.",
    "The Suez Canal connects the Mediterranean Sea to the Red Sea.",
    "Egypt is located in northeastern Africa and is home to the pyramids.",
    "Retrieval Augmented Generation improves LLM accuracy with retrieved context.",
    "BM25 is a probabilistic keyword ranking algorithm used in search engines.",
    "Semantic search uses embeddings to find documents by meaning, not keywords.",
    "The Nile River is the longest river in Africa.",
    "Cairo is the capital of Egypt and has a population of over 20 million.",
]

queries = [
    "How does RAG work?",
    "Tell me about Egypt and its rivers.",
    "What search algorithms exist for information retrieval?",
]

# YOUR CODE HERE - for each query, print the top-3 results with BM25 scores

tok_corpus = [tokenize(d) for d in corpus]
bm25_ex    = BM25Okapi(tok_corpus)

for q in queries:
    print(f"\nQuery: {q}")
    # YOUR CODE HERE - get scores, sort, print top 3
    pass

---
**Lab 02 complete.**

You built TF-IDF and BM25 from scratch and understand what each formula does.
BM25 is the standard keyword search algorithm - it's used inside Elasticsearch,
Weaviate's hybrid search, and most production search systems.

**Next:** `lab03_semantic_search.ipynb` - find documents by meaning, not keywords.